# Graph Programming — 05: Course Schedule & Topological Sort

## The Core Concept: Topological Sort

**Topological sort** orders nodes in a **Directed Acyclic Graph (DAG)** so that for every edge `u → v`, node `u` comes before `v`.

Real-world uses:
- **Course prerequisites** — you must take CS101 before CS201
- **Build systems** — compile module A before B
- **Task scheduling** — dependencies

```
Prerequisites:
  0 → 1 → 3
  0 → 2 → 3

Valid order: [0, 1, 2, 3]  or  [0, 2, 1, 3]
```

**If a cycle exists → no valid ordering → impossible.**

## Two Algorithms
1. **DFS-based** — uses `WHITE/GRAY/BLACK` coloring (or recursion state)
2. **Kahn's Algorithm (BFS-based)** — uses in-degree counting

---

## Problem 1: Course Schedule I (LC 207)

Given `numCourses` and a list of `prerequisites` pairs `[a, b]` (b must be taken before a),  
return `True` if you can finish all courses (i.e., no cycle exists).

### Approach A: DFS Cycle Detection

In [ ]:
from collections import defaultdict, deque

# ============================================================
# DFS CYCLE DETECTION with 3-color marking
# ============================================================
# WHITE (0) = unvisited
# GRAY  (1) = currently in DFS stack (ancestor)
# BLACK (2) = fully processed
#
# If you reach a GRAY node → cycle!

def canFinish_DFS(numCourses, prerequisites):
    graph = defaultdict(list)
    for course, prereq in prerequisites:
        graph[prereq].append(course)   # prereq → course

    # 0 = unvisited, 1 = in-progress, 2 = done
    state = [0] * numCourses

    def dfs(node):
        if state[node] == 1:   # currently in stack → cycle!
            return False
        if state[node] == 2:   # already processed → safe
            return True

        state[node] = 1        # mark as in-progress
        for neighbor in graph[node]:
            if not dfs(neighbor):
                return False
        state[node] = 2        # mark as fully done
        return True

    for course in range(numCourses):
        if not dfs(course):
            return False
    return True

print(canFinish_DFS(2, [[1,0]]))         # True  (0→1, no cycle)
print(canFinish_DFS(2, [[1,0],[0,1]]))   # False (0→1→0, cycle!)

### Approach B: Kahn's Algorithm (BFS + In-degree)

**In-degree** = how many edges point INTO a node = number of prerequisites.

```
Algorithm:
1. Calculate in-degree for all nodes
2. Enqueue all nodes with in-degree = 0 (no prerequisites)
3. Process queue:
   - Remove node, decrement in-degree of its neighbors
   - If neighbor's in-degree hits 0 → enqueue it
4. If we processed all nodes → no cycle. Otherwise → cycle.
```

In [ ]:
def canFinish_BFS(numCourses, prerequisites):
    graph = defaultdict(list)
    indegree = [0] * numCourses

    for course, prereq in prerequisites:
        graph[prereq].append(course)
        indegree[course] += 1

    # Start with all courses that have no prerequisites
    queue = deque([c for c in range(numCourses) if indegree[c] == 0])
    processed = 0

    while queue:
        course = queue.popleft()
        processed += 1
        for next_course in graph[course]:
            indegree[next_course] -= 1
            if indegree[next_course] == 0:   # all prereqs done!
                queue.append(next_course)

    return processed == numCourses  # if not all processed → cycle

print(canFinish_BFS(2, [[1,0]]))         # True
print(canFinish_BFS(2, [[1,0],[0,1]]))   # False
print(canFinish_BFS(4, [[1,0],[2,0],[3,1],[3,2]]))  # True

---
## Problem 2: Course Schedule II (LC 210)

Same as I, but return the **actual order** you can take the courses.  
If impossible (cycle), return `[]`.

In [ ]:
# === KAHN'S ALGORITHM — returns ordering ===

def findOrder_BFS(numCourses, prerequisites):
    graph = defaultdict(list)
    indegree = [0] * numCourses

    for course, prereq in prerequisites:
        graph[prereq].append(course)
        indegree[course] += 1

    queue = deque([c for c in range(numCourses) if indegree[c] == 0])
    order = []

    while queue:
        course = queue.popleft()
        order.append(course)          # record the order
        for next_course in graph[course]:
            indegree[next_course] -= 1
            if indegree[next_course] == 0:
                queue.append(next_course)

    return order if len(order) == numCourses else []

print(findOrder_BFS(2, [[1,0]]))                    # [0, 1]
print(findOrder_BFS(4, [[1,0],[2,0],[3,1],[3,2]]))  # [0,1,2,3] or [0,2,1,3]
print(findOrder_BFS(2, [[0,1],[1,0]]))              # [] (cycle)

In [ ]:
# === DFS — returns ordering (reverse postorder) ===

def findOrder_DFS(numCourses, prerequisites):
    graph = defaultdict(list)
    for course, prereq in prerequisites:
        graph[prereq].append(course)

    state = [0] * numCourses   # 0=unvisited, 1=in-stack, 2=done
    order = []
    has_cycle = [False]

    def dfs(node):
        if has_cycle[0]:
            return
        if state[node] == 1:
            has_cycle[0] = True
            return
        if state[node] == 2:
            return
        state[node] = 1
        for neighbor in graph[node]:
            dfs(neighbor)
        state[node] = 2
        order.append(node)   # add AFTER processing all descendants

    for c in range(numCourses):
        dfs(c)

    if has_cycle[0]:
        return []
    return order[::-1]   # reverse postorder = topological order

print(findOrder_DFS(2, [[1,0]]))                    # [0, 1]
print(findOrder_DFS(4, [[1,0],[2,0],[3,1],[3,2]]))  # [0,1,2,3] or [0,2,1,3]

---
## Problem 3: Alien Dictionary (Advanced)

Given a sorted list of words from an alien language, find the character ordering.

**Strategy:**
1. Compare adjacent words to find ordering constraints (character edges)
2. Build a directed graph from those constraints
3. Topological sort to find the character order

In [ ]:
def alienOrder(words):
    # Build adjacency list with all unique characters
    graph = {ch: set() for word in words for ch in word}
    indegree = {ch: 0 for ch in graph}

    # Compare adjacent pairs of words
    for i in range(len(words) - 1):
        w1, w2 = words[i], words[i+1]
        min_len = min(len(w1), len(w2))
        # Invalid: prefix word comes AFTER longer word
        if len(w1) > len(w2) and w1[:min_len] == w2[:min_len]:
            return ""
        for j in range(min_len):
            if w1[j] != w2[j]:
                if w2[j] not in graph[w1[j]]:
                    graph[w1[j]].add(w2[j])
                    indegree[w2[j]] += 1
                break  # only first different char gives info

    # Kahn's topological sort
    queue = deque([ch for ch in indegree if indegree[ch] == 0])
    result = []

    while queue:
        ch = queue.popleft()
        result.append(ch)
        for neighbor in graph[ch]:
            indegree[neighbor] -= 1
            if indegree[neighbor] == 0:
                queue.append(neighbor)

    return ''.join(result) if len(result) == len(graph) else ""

print(alienOrder(["wrt","wrf","er","ett","rftt"]))  # "wertf"
print(alienOrder(["z","x"]))                         # "zx"
print(alienOrder(["z","x","z"]))                     # "" (cycle)

---
## Side-by-Side Comparison

| | DFS Topo Sort | Kahn's (BFS) |
|---|---|---|
| Implementation | Recursive DFS + reverse postorder | BFS + in-degree counting |
| Cycle detection | GRAY node = cycle | `processed != n` = cycle |
| Order output | Reverse the append order | Append as you dequeue |
| Preferred when | Already thinking recursively | More intuitive, easier to debug |

**Kahn's is usually easier to code correctly in an interview.**

## The Universal Kahn's Template

```python
# 1. Build graph + compute in-degrees
graph = defaultdict(list)
indegree = defaultdict(int)
for u, v in edges:
    graph[u].append(v)
    indegree[v] += 1

# 2. Seed queue with zero in-degree nodes
queue = deque([n for n in all_nodes if indegree[n] == 0])
result = []

# 3. Process
while queue:
    node = queue.popleft()
    result.append(node)
    for neighbor in graph[node]:
        indegree[neighbor] -= 1
        if indegree[neighbor] == 0:
            queue.append(neighbor)

# 4. Cycle check
valid = len(result) == len(all_nodes)
```

**Next:** `06_advanced.ipynb` — Union Find, Dijkstra, Bipartite